In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import (
    train_test_split,
    GridSearchCV,
    StratifiedKFold
)

from sklearn.preprocessing import StandardScaler

from sklearn.pipeline import Pipeline

from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score
)

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier

from xgboost import XGBClassifier

import joblib

In [2]:
rfm = pd.read_csv("../data/processed/customer_features.csv")

rfm.head()

,Customer ID,Recency,Frequency,Monetary,AverageOrderValue,TotalQuantity,UniqueProducts,CustomerLifetime,AveragePurchaseGap,Churn
0,12346,325,1,77183.60,77183.600000,74215,1,0,0.0,1
1,12347,1,7,4310.00,23.681319,2458,103,365,2.0,0
2,12348,74,4,1797.24,57.975484,2341,22,282,9.4,0
3,12349,18,1,1757.55,24.076027,631,73,0,0.0,0
4,12350,309,1,334.40,19.670588,197,17,0,0.0,1


In [ ]:
#Separate Features and Target

In [3]:
X = rfm.drop(columns=["Customer ID", "Churn"])

y = rfm["Churn"]

In [ ]:
#Train-Test Split

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

In [5]:
print(X_train.shape)
print(X_test.shape)

(3470, 8)
(868, 8)


In [ ]:
#Scale Features (for Logistic Regression)

In [6]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
#Train Logistic Regression

In [7]:
lr = LogisticRegression(
    max_iter=1000,
    random_state=42
)

lr.fit(X_train_scaled, y_train)

lr_pred = lr.predict(X_test_scaled)

In [8]:
print("Accuracy :", accuracy_score(y_test, lr_pred))
print("Precision:", precision_score(y_test, lr_pred))
print("Recall   :", recall_score(y_test, lr_pred))
print("F1 Score :", f1_score(y_test, lr_pred))
print("ROC AUC  :", roc_auc_score(y_test, lr.predict_proba(X_test_scaled)[:,1]))

Accuracy : 0.988479262672811
Precision: 1.0
Recall   : 0.9653979238754326
F1 Score : 0.9823943661971831
ROC AUC  : 1.0


In [ ]:
#Train Decision Tree

In [9]:
dt = DecisionTreeClassifier(random_state=42)

dt.fit(X_train, y_train)

dt_pred = dt.predict(X_test)

In [10]:
print("Accuracy :", accuracy_score(y_test, dt_pred))
print("Precision:", precision_score(y_test, dt_pred))
print("Recall   :", recall_score(y_test, dt_pred))
print("F1 Score :", f1_score(y_test, dt_pred))
print("ROC AUC  :", roc_auc_score(y_test, dt.predict_proba(X_test)[:,1]))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC AUC  : 1.0


In [ ]:
#Train Random Forest

In [11]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42
)

rf.fit(X_train, y_train)

rf_pred = rf.predict(X_test)

In [13]:
print("Accuracy :", accuracy_score(y_test, rf_pred))
print("Precision:", precision_score(y_test, rf_pred))
print("Recall   :", recall_score(y_test, rf_pred))
print("F1 Score :", f1_score(y_test, rf_pred))
print("ROC AUC  :", roc_auc_score(y_test, rf.predict_proba(X_test)[:,1]))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC AUC  : 1.0


In [ ]:
#Train XGBoost

In [14]:
xgb = XGBClassifier(
    random_state=42,
    eval_metric="logloss"
)

xgb.fit(X_train, y_train)

xgb_pred = xgb.predict(X_test)

In [15]:
print("Accuracy :", accuracy_score(y_test, xgb_pred))
print("Precision:", precision_score(y_test, xgb_pred))
print("Recall   :", recall_score(y_test, xgb_pred))
print("F1 Score :", f1_score(y_test, xgb_pred))
print("ROC AUC  :", roc_auc_score(y_test, xgb.predict_proba(X_test)[:,1]))

Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC AUC  : 1.0


In [ ]:
#Compare Models

In [16]:
models = {
    "Logistic Regression": (lr_pred, lr.predict_proba(X_test_scaled)[:,1]),
    "Decision Tree": (dt_pred, dt.predict_proba(X_test)[:,1]),
    "Random Forest": (rf_pred, rf.predict_proba(X_test)[:,1]),
    "XGBoost": (xgb_pred, xgb.predict_proba(X_test)[:,1])
}

results = []

for name, (pred, prob) in models.items():
    results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1 Score": f1_score(y_test, pred),
        "ROC AUC": roc_auc_score(y_test, prob)
    })

results_df = pd.DataFrame(results)
results_df.sort_values(by="ROC AUC", ascending=False)

,Model,Accuracy,Precision,Recall,F1 Score,ROC AUC
0,Logistic Regression,0.988479,1.0,0.965398,0.982394,1.0
1,Decision Tree,1.000000,1.0,1.000000,1.000000,1.0
2,Random Forest,1.000000,1.0,1.000000,1.000000,1.0
3,XGBoost,1.000000,1.0,1.000000,1.000000,1.0


In [ ]:
#Hyperparameter Tuning (Random Forest Example)

In [ ]:
param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20],
    "min_samples_split": [2, 5]
}

grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring="roc_auc",
    n_jobs=-1
)

grid.fit(X_train, y_train)

best_rf = grid.best_estimator_

print("Best Parameters:", grid.best_params_)

In [ ]:
#Save the Best Model

In [ ]:
joblib.dump(best_rf, "../models/churn_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

In [13]:
import os

print(os.listdir("../src"))

['.ipynb_checkpoints', 'data_preprocessing.py', 'feature_engineering.py', 'predict.py', 'train_model.py', 'utils.py', '__init__.py', '__pycache__']


In [14]:
import sys
import os

project_root = os.path.abspath("..")
sys.path.append(project_root)

print(project_root)

C:\Users\shant\Python\customer-retention-system


In [15]:
from src.train_model import train_models

In [16]:
train_models("../data/processed/customer_features.csv")

MODEL PERFORMANCE

Logistic Regression
------------------------------
Accuracy : 0.988479262672811
Precision: 1.0
Recall   : 0.9653979238754326
F1 Score : 0.9823943661971831
ROC AUC  : 1.0

Decision Tree
------------------------------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC AUC  : 1.0

Random Forest
------------------------------
Accuracy : 1.0
Precision: 1.0
Recall   : 1.0
F1 Score : 1.0
ROC AUC  : 1.0

Best Model Saved Successfully!


,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'
